# Calculate NDWI from STAC Imagery

This notebook computes the **Normalized Difference Water Index (NDWI)** from optical imagery served via STAC. It works with **Platero L1C** (per-band G and NIR COGs, with scale/offset) and **Sentinel-2** (B03/B08 or a single multi-band COG). Public catalogs can be used without a token; authenticated data (e.g. Platero) require a token in `.env`.

---

## What is NDWI?

NDWI uses the difference between green and near-infrared (NIR) reflectance to detect water and surface moisture. Values range from **−1** to **+1**:

| Range   | Interpretation        |
|--------|------------------------|
| ≈ 1    | Water bodies (open water, lakes, rivers) |
| ≈ 0    | Bare soil or sparse vegetation |
| ≈ −1   | Dense vegetation or dry surfaces |

## Parameters

- **STAC Item & Collection** — Set in the *Set variables* cell. Use the placeholders `{{STAC_ITEM_LINK}}` and `{{STAC_COLLECTION_NAME}}` when running from a template.
- **Token** — Optional. Required only for authenticated catalogs/COGs; store as `token=...` in a `.env` file in this directory.
- **AOI** — Optional GeoJSON geometry. If empty, a 1000×1000 pixel window from the centre of the image is used.

## Workflow

1. Load the STAC item (using token if provided).
2. Resolve green and NIR bands (Platero G/NIR, Sentinel-2 B03/B08, or multi-band COG).
3. Read band data and apply scale/offset when using per-band assets.
4. Compute NDWI: **(Green − NIR) / (Green + NIR)**.
5. Visualise the results.

## Import Required Libraries

In [ ]:
%pip install python-dotenv

In [ ]:
import os
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
from rasterio.warp import transform_geom
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import json
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## Authorisation

For **non-public** datasets (e.g. Platero), provide a token in a `.env` file in this directory:

```bash
token=your_token_here
```

See [EODH documentation on sensitive data](https://eodatahub.org.uk/docs/documentation/notebooks/sensitive-data/) for details. Public STAC catalogues do not require a token.

In [ ]:
token = os.getenv("token")
# So rasterio/GDAL send the token when opening COG URLs (avoids 401 on asset hrefs)
if token:
    os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {token}"

## Set variables

Set the STAC item URL and collection name for your scene. When launched from a template, the placeholders below may already be filled.

In [ ]:
stac_item_url = "{{STAC_ITEM_LINK}}"
stac_collection_name = "{{STAC_COLLECTION_NAME}}"
aoi_param = """{{AOI}}""".strip()

## Load STAC item

In [ ]:
try:
    if token:
        stac_io = pystac.StacIO.default()
        stac_io.headers = {"Authorization": f"Bearer {token}"}
        item = pystac.Item.from_file(stac_item_url, stac_io=stac_io)
    else:
        item = pystac.Item.from_file(stac_item_url)
    print(f"Successfully loaded STAC item: {item.id}")
    print(f"Collection: {stac_collection_name}")
    print(f"Date: {item.datetime}")
except Exception as e:
    print(f"Error loading STAC item: {e}")
    raise

## Determine Area of Interest (AOI)

Define the region to process. If you provide a **GeoJSON** geometry (e.g. Polygon or FeatureCollection), the raster is clipped to it. Otherwise, a **1000×1000 pixel** window is taken from the centre of the image.

In [ ]:
DEFAULT_WINDOW_SIZE = 1000

aoi_geometry = None
clip_window = None
use_windowed_read = False

if aoi_param and aoi_param.strip().lower() not in ("", "none", "null"):
    try:
        # Parse as JSON
        aoi_data = json.loads(aoi_param)

        # Extract geometry from GeoJSON structure
        if aoi_data.get("type") == "FeatureCollection":
            # Extract first geometry from FeatureCollection
            if aoi_data.get("features") and len(aoi_data["features"]) > 0:
                aoi_geometry = aoi_data["features"][0].get("geometry")
        elif aoi_data.get("type") == "Feature":
            # Extract geometry from Feature
            aoi_geometry = aoi_data.get("geometry")
        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:
            # Direct geometry object
            aoi_geometry = aoi_data
        else:
            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")

        # Validate geometry was extracted
        if aoi_geometry and aoi_geometry.get("type"):
            print(f"AOI provided: {aoi_geometry['type']} geometry")
            print("Will clip raster to AOI geometry")
        else:
            raise ValueError("Could not extract geometry from GeoJSON")

    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Warning: Could not parse AOI: {e}. Using default window.")
        aoi_geometry = None
else:
    print("No AOI provided, using default 1000×1000 pixel window")
if aoi_geometry is None:
    use_windowed_read = True
    print(
        f"Will extract {DEFAULT_WINDOW_SIZE}×{DEFAULT_WINDOW_SIZE} pixel window from center"
    )

In [ ]:
# STAC item and variables are set in the cells above.

## Access Green and NIR Bands

The notebook automatically detects how green and NIR are provided:

- **Platero L1C** — Per-band assets **G** and **NIR**; scale and offset from STAC are applied when reading.
- **Sentinel-2** — Per-band B03/B08, or a single multi-band COG (asset key `cog`, `data`, or `image`) with `eo:bands` describing band order.

In [ ]:
def get_scale_offset(asset):
    """Read scale and offset from asset bands metadata (e.g. Platero). Returns (1.0, 0.0) if absent."""
    extra = _asset_extra(asset)
    bands = extra.get("bands", []) or extra.get("raster:bands", [])
    if bands:
        b = bands[0]
        return float(b.get("scale", 1.0)), float(b.get("offset", 0.0))
    return 1.0, 0.0


def _norm(text):
    return str(text or "").strip().lower()


def _asset_extra(asset):
    if isinstance(asset, dict):
        return asset
    return getattr(asset, "extra_fields", None) or getattr(asset, "extra", None) or {}


def _asset_band_entries(asset):
    extra = _asset_extra(asset)
    return extra.get("eo:bands") or extra.get("bands") or []


try:
    green_band_asset = None
    nir_band_asset = None
    nir08_band_asset = None
    cog_asset = None
    green_band_index = 3
    nir_band_index = 8
    green_scale, green_offset = 1.0, 0.0
    nir_scale, nir_offset = 1.0, 0.0

    # 1) Per-band assets: metadata-first, with key-name fallbacks.
    for asset_key, asset in item.assets.items():
        key = _norm(asset_key)
        eo_bands = _asset_band_entries(asset)

        green_match = False
        nir_match = False
        nir08_match = False

        for b in eo_bands:
            common_name = _norm(b.get("common_name") or b.get("eo:common_name"))
            name = _norm(b.get("name") or b.get("eo:name"))

            if common_name == "green" or name == "b03":
                green_match = True
            if common_name == "nir" or name == "b08":
                nir_match = True
            if common_name == "nir08" or name == "b8a":
                nir08_match = True

        if not green_match:
            green_match = (
                key in {"g", "green", "b03"}
                or "b03" in key
                or (key.startswith("green") and "visual" not in key and "preview" not in key)
            )
        if not nir_match:
            nir_match = key in {"nir", "b08"} or "b08" in key
        if not nir08_match:
            nir08_match = key in {"nir08", "b8a"} or "b8a" in key

        is_single_band = len(eo_bands) <= 1
        if is_single_band and green_match and green_band_asset is None:
            green_band_asset = asset
            print(f"Found green band: {asset_key}")
        if is_single_band and nir_match and nir_band_asset is None:
            nir_band_asset = asset
            print(f"Found NIR band (B08): {asset_key}")
        if is_single_band and nir08_match and nir08_band_asset is None:
            nir08_band_asset = asset
            print(f"Found NIR narrow band (B8A): {asset_key}")

    if nir_band_asset is None and nir08_band_asset is not None:
        nir_band_asset = nir08_band_asset
        print("Using B8A/nir08 as NIR fallback (B08 not found).")

    # 2) If no per-band pair, look for multi-band COG/stacked assets.
    if green_band_asset is None or nir_band_asset is None:
        preferred_multi_band_keys = ["cog", "data", "image", "reflectance", "bands"]
        candidate_keys = preferred_multi_band_keys + [
            k for k in item.assets.keys() if k not in preferred_multi_band_keys
        ]

        for asset_key in candidate_keys:
            if asset_key not in item.assets:
                continue

            candidate_asset = item.assets[asset_key]
            eo_bands = _asset_band_entries(candidate_asset)
            if len(eo_bands) < 2:
                continue

            green_idx = None
            nir_idx = None
            nir08_idx = None

            for i, b in enumerate(eo_bands, start=1):
                common_name = _norm(b.get("common_name") or b.get("eo:common_name"))
                name = _norm(b.get("name") or b.get("eo:name"))
                if green_idx is None and (common_name == "green" or name == "b03"):
                    green_idx = i
                if nir_idx is None and (common_name == "nir" or name == "b08"):
                    nir_idx = i
                if nir08_idx is None and (common_name == "nir08" or name == "b8a"):
                    nir08_idx = i

            if green_idx is None:
                continue
            if nir_idx is None and nir08_idx is None:
                continue

            cog_asset = candidate_asset
            green_band_index = green_idx
            nir_band_index = nir_idx if nir_idx is not None else nir08_idx
            selected_nir_label = "NIR" if nir_idx is not None else "NIR (B8A fallback)"
            print(
                f"Using multi-band asset '{asset_key}' (green=band {green_band_index}, {selected_nir_label}=band {nir_band_index})"
            )
            break

        if cog_asset is None:
            raise ValueError(
                "Could not find green or NIR bands. "
                "Need per-band G/NIR (Platero) or B03/B08, or a multi-band 'cog'/'data' asset."
            )

    if green_band_asset is not None and nir_band_asset is not None:
        green_scale, green_offset = get_scale_offset(green_band_asset)
        nir_scale, nir_offset = get_scale_offset(nir_band_asset)
        print("\nPer-band assets found.")
        print(f"Green href: {green_band_asset.href}")
        print(f"NIR href: {nir_band_asset.href}")
        if (
            green_scale != 1.0
            or green_offset != 0.0
            or nir_scale != 1.0
            or nir_offset != 0.0
        ):
            print(
                f"Scale/offset will be applied: G ({green_scale}, {green_offset}), NIR ({nir_scale}, {nir_offset})"
            )
    else:
        print(f"\nMulti-band COG href: {cog_asset.href}")

except Exception as e:
    print(f"Error accessing bands: {e}")
    print("Available assets:", list(item.assets.keys()))
    raise


## Read Band Data

Read the green and NIR layers—either from a multi-band COG (by band index) or from separate band assets. Data are clipped to the AOI or to the central 1000×1000 window if no AOI was set. For per-band assets (e.g. Platero), scale and offset from the STAC metadata are applied to obtain physical values.

In [ ]:
try:
    # Ensure AOI variables are initialized (in case AOI cell was not executed)
    try:
        _ = aoi_geometry
    except NameError:
        aoi_geometry = None
        clip_window = None
        use_windowed_read = False

    if cog_asset is not None:
        # Read B03 and B08 from multi-band COG (1-based band indices)
        with rasterio.open(cog_asset.href) as src:
            if src.count < max(green_band_index, nir_band_index):
                raise ValueError(
                    f"COG has {src.count} bands; need at least band {max(green_band_index, nir_band_index)}. "
                    "Check band order (e.g. B01,B02,B03,B04,...)."
                )

            # Determine clip window or use default window
            if aoi_geometry is not None:
                # Reproject AOI to raster CRS (GeoJSON is typically WGS84)
                aoi_crs = "EPSG:4326"
                aoi_in_raster_crs = transform_geom(aoi_crs, src.crs, aoi_geometry)
                # Clip to AOI geometry
                green_data, green_transform = mask(
                    src, [aoi_in_raster_crs], crop=True, indexes=[green_band_index]
                )
                nir_data, nir_transform = mask(
                    src, [aoi_in_raster_crs], crop=True, indexes=[nir_band_index]
                )
                green_data = green_data[0]  # Remove band dimension
                nir_data = nir_data[0]
                # Update profile with clipped bounds
                green_profile = src.profile.copy()
                green_profile.update(
                    {
                        "height": green_data.shape[0],
                        "width": green_data.shape[1],
                        "transform": green_transform,
                        "count": 1,
                    }
                )
                print(f"Clipped to AOI geometry: {green_data.shape}")
            elif use_windowed_read:
                # Extract center 1000x1000 pixel window
                height, width = src.height, src.width
                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                    print(
                        f"Image is smaller than {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE}, using full image"
                    )
                    clip_window = None
                else:
                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                    clip_window = Window(
                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                    )
                    print(
                        f"Extracting {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"
                    )

                green_data = src.read(green_band_index, window=clip_window)
                nir_data = src.read(nir_band_index, window=clip_window)
                green_profile = src.profile.copy()
                if clip_window:
                    green_profile.update(
                        {
                            "height": clip_window.height,
                            "width": clip_window.width,
                            "transform": rasterio.windows.transform(
                                clip_window, src.transform
                            ),
                            "count": 1,
                        }
                    )
                else:
                    green_profile.update(count=1)
            else:
                # Read full image
                green_data = src.read(green_band_index)
                nir_data = src.read(nir_band_index)
                green_profile = src.profile.copy()
                green_profile.update(count=1)

            green_crs = src.crs
            print(
                f"Read from multi-band COG: band {green_band_index} (green), band {nir_band_index} (NIR)"
            )
    else:
        # Read from separate band assets
        with rasterio.open(green_band_asset.href) as green_src:
            if aoi_geometry is not None:
                aoi_crs = "EPSG:4326"
                aoi_in_raster_crs = transform_geom(aoi_crs, green_src.crs, aoi_geometry)
                green_data, green_transform = mask(
                    green_src, [aoi_in_raster_crs], crop=True
                )
                green_data = green_data[0]
                green_profile = green_src.profile.copy()
                green_profile.update(
                    {
                        "height": green_data.shape[0],
                        "width": green_data.shape[1],
                        "transform": green_transform,
                        "count": 1,
                    }
                )
            elif use_windowed_read:
                height, width = green_src.height, green_src.width
                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                    clip_window = None
                else:
                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                    clip_window = Window(
                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                    )
                green_data = green_src.read(1, window=clip_window)
                green_profile = green_src.profile.copy()
                if clip_window:
                    green_profile.update(
                        {
                            "height": clip_window.height,
                            "width": clip_window.width,
                            "transform": rasterio.windows.transform(
                                clip_window, green_src.transform
                            ),
                        }
                    )
            else:
                green_data = green_src.read(1)
                green_profile = green_src.profile.copy()
            green_crs = green_src.crs

        with rasterio.open(nir_band_asset.href) as nir_src:
            if aoi_geometry is not None:
                aoi_crs = "EPSG:4326"
                aoi_in_raster_crs = transform_geom(aoi_crs, nir_src.crs, aoi_geometry)
                nir_data, nir_transform = mask(nir_src, [aoi_in_raster_crs], crop=True)
                nir_data = nir_data[0]
            elif use_windowed_read:
                nir_data = nir_src.read(1, window=clip_window)
            else:
                nir_data = nir_src.read(1)
            # Use same profile as green (same grid)
            green_profile = nir_src.profile.copy()
            if aoi_geometry is not None:
                green_profile.update(
                    {
                        "height": nir_data.shape[0],
                        "width": nir_data.shape[1],
                        "transform": nir_transform,
                        "count": 1,
                    }
                )
            elif clip_window:
                green_profile.update(
                    {
                        "height": clip_window.height,
                        "width": clip_window.width,
                        "transform": rasterio.windows.transform(
                            clip_window, nir_src.transform
                        ),
                    }
                )

    print(f"Green band shape: {green_data.shape}, dtype: {green_data.dtype}")
    print(f"NIR band shape: {nir_data.shape}, dtype: {nir_data.dtype}")
    print(f"CRS: {green_crs}")

    if green_data.shape != nir_data.shape:
        raise ValueError(
            f"Band shapes do not match: Green {green_data.shape} vs NIR {nir_data.shape}"
        )

    if cog_asset is not None:
        green_data = green_data.astype(np.float32)
        nir_data = nir_data.astype(np.float32)
    else:
        green_data = green_data.astype(np.float32) * green_scale + green_offset
        nir_data = nir_data.astype(np.float32) * nir_scale + nir_offset
    print(f"\nBand data loaded successfully! Processing {green_data.size:,} pixels")

except Exception as e:
    print(f"Error reading band data: {e}")
    raise

## Calculate NDWI

$$NDWI = \frac{Green - NIR}{Green + NIR}$$

Valid pixels (where the denominator is non-zero) are computed; invalid pixels are set to NaN. Output values are clamped to the range **[−1, 1]**.

In [ ]:
denominator = green_data + nir_data
valid_mask = denominator != 0
ndwi = np.full_like(green_data, np.nan, dtype=np.float32)
ndwi[valid_mask] = (green_data[valid_mask] - nir_data[valid_mask]) / denominator[
    valid_mask
]
ndwi = np.clip(ndwi, -1.0, 1.0)

print("NDWI calculation complete!")
print(
    f"NDWI min: {np.nanmin(ndwi):.4f}, max: {np.nanmax(ndwi):.4f}, mean: {np.nanmean(ndwi):.4f}"
)
print(f"Valid pixels: {np.sum(~np.isnan(ndwi)):,} of {ndwi.size:,}")

## Visualise Results

Green and NIR bands are displayed with a percentile-based stretch (98th percentile) for contrast. The NDWI panel uses a colour scale from dense vegetation (brown) to water (blue).

In [ ]:
# Create a custom colormap for NDWI visualization
# Colors: dense vegetation (brown) -> sparse vegetation (yellow) -> bare soil (gray) -> water (blue)
colors = ["#8B4513", "#D2691E", "#CCCCCC", "#87CEEB", "#4169E1", "#000080"]
n_bins = 256
cmap = LinearSegmentedColormap.from_list("ndwi", colors, N=n_bins)

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax_green = (
    np.percentile(green_data[~np.isnan(green_data) & (green_data > 0)], 98)
    if np.any(green_data > 0)
    else np.nanmax(green_data)
)
vmax_nir = (
    np.percentile(nir_data[~np.isnan(nir_data) & (nir_data > 0)], 98)
    if np.any(nir_data > 0)
    else np.nanmax(nir_data)
)
axes[0].imshow(green_data, cmap="Greens", vmin=0, vmax=vmax_green)
axes[0].set_title("Green Band", fontsize=14, fontweight="bold")
axes[0].axis("off")
plt.colorbar(
    axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.04, label="Reflectance"
)

axes[1].imshow(nir_data, cmap="YlGn", vmin=0, vmax=vmax_nir)
axes[1].set_title("NIR Band", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.colorbar(
    axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04, label="Reflectance"
)

im3 = axes[2].imshow(ndwi, cmap=cmap, vmin=-1, vmax=1)
axes[2].set_title("NDWI", fontsize=14, fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="NDWI")

# Add colorbar labels
cbar.set_ticks([-1, -0.5, 0, 0.3, 0.6, 1])
cbar.set_ticklabels(
    [
        "Dense Veg",
        "Sparse Veg",
        "Bare Soil",
        "Moist Soil",
        "Shallow Water",
        "Deep Water",
    ]
)

plt.suptitle(f"NDWI — {item.id}", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Visualisation complete.")

## Summary

- **Supported data**: Platero L1C (G/NIR per-band COGs) and Sentinel-2 (B03/B08 or multi-band COG).
- **Auth**: Use a token in `.env` only for authenticated catalogues; public data work without it.
- **Steps**: Load STAC item → resolve green and NIR bands → read data (with scale/offset for per-band assets) → compute NDWI → visualise.
- **Next**: Change `stac_item_url` and `stac_collection_name` for other scenes; optionally set `aoi_param` to a GeoJSON geometry.